In [1]:
import polars as pl
import numpy as np
import pandas as pd

In [2]:
data = pl.read_csv('./data/listings_preprocessed.csv')
df = pl.DataFrame(data)

## Обработка пропусков

Стратегия: **не удаляем строки**, только заполняем пропуски.
Сначала приводим пустые строки и `'nan'` к `null`, затем обрабатываем каждый столбец отдельно — от столбцов с наименьшим числом пропусков к наибольшему.

In [3]:
df.null_count()

cian_id,first_seen,last_seen,sold,misses,days_on_market,price,price_per_m2,deal_conditions,region,municipality,district,rooms,total_area,living_area,kitchen_area,floor,total_floors,ceiling_height,renovation,bathrooms,balcony,window_view,is_apartments,year_built,building_type,parking,is_new_building,developer,residential_complex,completion_date,seller_type,phone_protected,publication_date,price_first,price_last
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,2601,0,61,60627,356,0,45495,47617,0,0,35053,110423,58212,123345,87248,15118,118702,26165,81034,0,53083,52811,85623,19871,0,0,0,0


In [4]:
col_with_null = [
    'municipality', 'deal_conditions', 'rooms', 'is_apartments', 'seller_type',
    'building_type', 'ceiling_height', 'kitchen_area', 'living_area',
    'residential_complex', 'developer', 'bathrooms', 'district', 'parking',
    'window_view', 'completion_date', 'renovation', 'year_built', 'balcony'
]

# пропуски и текстовые NaN в null
df = df.with_columns([
    pl.when(
        pl.col(c).is_null() | (pl.col(c).cast(pl.Utf8).str.strip_chars().str.to_lowercase().is_in(['', 'nan', 'none']))
    ).then(None).otherwise(pl.col(c)).alias(c)
    for c in col_with_null
])

print(f'Строк в датасете: {df.height}')
print('\nПропуски по столбцам (до обработки):')
nulls = df.select([pl.col(c).null_count().alias(c) for c in col_with_null])
nulls.transpose(include_header=True, header_name='column', column_names=['missing']).sort('missing', descending=True)

Строк в датасете: 198106

Пропуски по столбцам (до обработки):


column,missing
str,u32
"""balcony""",123345
"""year_built""",118702
"""renovation""",110423
"""window_view""",87248
"""completion_date""",85623
…,…
"""seller_type""",19871
"""is_apartments""",15118
"""deal_conditions""",2601


### municipality

In [5]:
# заполняем пропуски Unknown
df = df.with_columns(pl.col("municipality").fill_null('Unknown').alias('municipality'))

### deal_conditions

In [6]:
# Определяем моду для deal_conditions
deal_conditions_mode = df.select(pl.col('deal_conditions').mode()).to_series(0)[0]
df = df.with_columns(pl.col('deal_conditions').fill_null(deal_conditions_mode).alias('deal_conditions'))

### rooms

Оцениваем число комнат по общей площади total_area

In [7]:
df.group_by("rooms").agg(pl.col("total_area").mean())

rooms,total_area
f64,f64
null,37.738933
2.0,57.042191
3.0,77.722483
5.0,122.552924
4.0,99.594761
1.0,39.481399
6.0,130.2325


In [8]:
df = df.with_columns(
    pl.when(pl.col('rooms').is_null())
    .then(
        pl.when(pl.col('total_area') <= 30).then(0)
        .when(pl.col('total_area') <= 40).then(1.0)
        .when(pl.col('total_area') <= 55).then(2.0)
        .when(pl.col('total_area') <= 80).then(3.0)
        .when(pl.col('total_area') <= 110).then(4.0)
        .otherwise(5.0)
    )
    .otherwise(pl.col('rooms'))
    .alias('rooms')
)

### is_apartments

In [9]:
# Постсчет соотношения апартаментов и не апартаментов
num_is_apartments = df['is_apartments'].sum()
num_no_apartments = (~df['is_apartments']).sum()
print(f'Апартаменты: {num_is_apartments} ({num_is_apartments / len(df):.2%})')
print(f'Не апартаменты: {num_no_apartments} ({num_no_apartments / len(df):.2%})')

Апартаменты: 7938 (4.01%)
Не апартаменты: 175050 (88.36%)


In [10]:
# Заполняем False, т.к. апартаменты встречаются реже обычных квартир.
df = df.with_columns(pl.col('is_apartments').fill_null(False).alias('is_apartments'))

### seller_type

In [11]:
# Находим моду по seller_type
seller_type_mode = (
    df.filter(pl.col('seller_type').is_not_null()).select(pl.col('seller_type').mode().first()).item()
)

# Заполняем модой
df = df.with_columns(
    pl.col('seller_type').fill_null('agency').alias('seller_type')
)

### building_type

In [12]:
# Заполняем пропуски на Unknown
df = df.with_columns(pl.col('building_type').fill_null('Unknown').alias('building_type'))

### ceiling_height

In [13]:
# Группируем по building_type и вычисляем медиану ceiling_height для каждого типа здания
ch_bt = (
    df.filter(pl.col('ceiling_height').is_not_null())
      .group_by('building_type')
      .agg(pl.col('ceiling_height').median().alias('ceiling_height'))
)

# Получаем глобальную медиану по ceiling_height
ch_median = (
    df.filter(pl.col('ceiling_height').is_not_null())
    .select(pl.col('ceiling_height').median())
    .item()
)

# Заполняем пропуски сначала медианой по типу здания, затем общей медианой
df = df.with_columns(
    pl.col('ceiling_height')
    .fill_null(ch_median)
    .alias('ceiling_height')
)

### kitchen_area и living_area

In [14]:
unknown_kitchen_living = df.filter(
    pl.col('kitchen_area').is_null() & pl.col('living_area').is_null()
)
unknown_kitchen_living.shape[0]

25532

In [15]:
# Группируем по building_type и total_area для kitchen_area
kitchen_bt = (
    df.filter(pl.col('kitchen_area').is_not_null())
      .group_by(['building_type', 'total_area'])
      .agg(pl.col('kitchen_area').median().alias('kitchen_area_med'))
)
# Группируем по building_type и total_area для living_area
living_bt = (
    df.filter(pl.col('living_area').is_not_null())
      .group_by(['building_type', 'total_area'])
      .agg(pl.col('living_area').median().alias('living_area_med'))
)

# Получаем глобальные медианы по kitchen_area и living_area
kitchen_global_median = df.filter(
    pl.col('kitchen_area').is_not_null()).select(pl.col('kitchen_area').median()
	).item()
living_global_median = df.filter(
    pl.col('living_area').is_not_null()).select(pl.col('living_area').median()
	).item()

# Присоединяем медианы групп и заполняем пропуски ими, потом глобальной медианой
df = (
    df.join(kitchen_bt, on=['building_type', 'total_area'], how='left')
      .join(living_bt, on=['building_type', 'total_area'], how='left')
      .with_columns([
          pl.when(pl.col('kitchen_area').is_null())
            .then(pl.col('kitchen_area_med'))
            .otherwise(pl.col('kitchen_area'))
            .fill_null(kitchen_global_median)
            .alias('kitchen_area'),
          pl.when(pl.col('living_area').is_null())
            .then(pl.col('living_area_med'))
            .otherwise(pl.col('living_area'))
            .fill_null(living_global_median)
            .alias('living_area'),
      ])
      .drop(['kitchen_area_med', 'living_area_med'])
)

### developer и residential_complex

In [16]:
# Заменяем пропуски на Unknown
df = df.with_columns(pl.col('developer').fill_null('Unknown').alias('developer'))
df = df.with_columns(pl.col('residential_complex').fill_null('Unknown').alias('residential_complex'))

### bathrooms

In [17]:
# Мода по столбцу bathrooms
bathrooms_mode = df.filter(
    pl.col('bathrooms').is_not_null()).select(pl.col('bathrooms').mode().first()
    ).item()
# Заполняем пропуски
df = df.with_columns(
    pl.col('bathrooms').fill_null(bathrooms_mode).alias('bathrooms')
)

### district

In [18]:
df = df.with_columns(pl.col('district').fill_null('Unknown').alias('district'))

### parking

In [19]:
df = df.with_columns(pl.col('parking').fill_null('Unknown').alias('parking'))

### window_view

In [20]:
df = df.with_columns(pl.col('window_view').fill_null('Unknown').alias('window_view'))

### completion_date

In [21]:
# Для вторички is_new_building = False - Сдан.  
# Для новостроек без даты - Unknown.
df = df.with_columns(
    pl.when(pl.col('completion_date').is_null() & ~pl.col('is_new_building'))
    .then(pl.lit('Сдан'))
    .when(pl.col('completion_date').is_null() & pl.col('is_new_building'))
    .then(pl.lit('Unknown'))
    .otherwise(pl.col('completion_date'))
    .alias('completion_date')
)

### renovation

In [22]:
df = df.with_columns(pl.col('renovation').fill_null('Unknown').alias('renovation'))

### year_built

In [23]:
# используем только медиану по муниципалитету

df = df.with_columns(pl.col('year_built').cast(pl.Int32).alias('year_built'))

median_years = df.group_by('municipality').agg(pl.col('year_built').median().alias('median_year_built'))
df = df.join(median_years, on='municipality', how='left')
df = df.with_columns(
    pl.when(pl.col('year_built').is_null())
      .then(pl.col('median_year_built'))
      .otherwise(pl.col('year_built'))
      .alias('year_built')
)

df = df.drop('median_year_built')

In [24]:
df['year_built'].null_count()

114

- Пропуски остаются из-за того, что д ля заполнения пропущенных значений используется медиана по муниципалитету, но если в каком-то муниципалитете у всех строк year_built=null, то медиана тоже будет null и, соответственно, эти пропущенные значения останутся незаполненными. Что бы не искажать значения удалим данные строки, т.к. степень их влияния на результат `крайне мала`

In [25]:
df = df.drop_nulls('year_built')

### balcony

In [26]:
df = df.with_columns(pl.col('balcony').fill_null('Unknown').alias('balcony'))

### Итоговая проверка

In [27]:
display(df.null_count())
print('Размер датасета', df.shape)
df.write_csv('./data/without_null_data.csv')

cian_id,first_seen,last_seen,sold,misses,days_on_market,price,price_per_m2,deal_conditions,region,municipality,district,rooms,total_area,living_area,kitchen_area,floor,total_floors,ceiling_height,renovation,bathrooms,balcony,window_view,is_apartments,year_built,building_type,parking,is_new_building,developer,residential_complex,completion_date,seller_type,phone_protected,publication_date,price_first,price_last
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Размер датасета (197992, 36)
